# 12. Análisis de errores e interpretación

Este análisis describe cómo se comportan las probabilidades finales en el test 2017–2018 y en qué segmentos aparecen falsos positivos y falsos negativos. Usa las predicciones calibradas ya guardadas por `11_evaluacion_final.ipynb`; **no vuelve a entrenar, seleccionar modelos ni ajustar umbrales con el test**.

Para las métricas que requieren una decisión binaria se fija el umbral convencional de **0.50**, sin optimizarlo en el test. Las métricas ROC-AUC y PR-AUC usan las probabilidades continuas. El test solo incluye `Fully Paid` y `Charged Off`; los préstamos sin desenlace no forman parte de estas tablas.

## 1. Cargar y comprobar predicciones y segmentos

Se leen del CSV original solo el año, el plazo, el ingreso y el tipo de solicitud de los préstamos con estado final desde 2017. Antes de unir esas columnas con las predicciones, se comprueba que el número, orden, año y etiqueta coincidan. No se guardan identificadores ni datos fila a fila en las tablas del informe.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.analisis_errores import ejecutar_analisis
from src.config import localizar_datos

ruta_datos = localizar_datos()
predicciones = RAIZ / "outputs/evaluacion_final/predicciones_test.csv.gz"
datos_error, metricas_globales, metricas_segmentos = ejecutar_analisis(
    ruta=ruta_datos, archivo_predicciones=predicciones,
    salida=RAIZ / "outputs/analisis_errores"
)
print(f"Filas verificadas: {len(datos_error):,}; años: {sorted(datos_error.anio.unique())}")
display(metricas_globales.round(4))

## 2. Falsos positivos y falsos negativos por segmento

Un falso positivo es un préstamo pagado que el modelo marca por encima de 0.50; un falso negativo es un `Charged Off` cuya probabilidad queda por debajo de 0.50. Los segmentos de ingreso son rangos descriptivos. Las diferencias no prueban que una característica cause el error ni, por sí solas, que exista discriminación.

In [ ]:
display(metricas_segmentos[metricas_segmentos["modelo"] == "gradient_boosting"]
        .sort_values(["segmento", "grupo"]).round(4))

In [ ]:
hgb = metricas_segmentos[metricas_segmentos["modelo"] == "gradient_boosting"]
por_anio = hgb[hgb["segmento"] == "anio"].sort_values("grupo")
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(por_anio["grupo"], por_anio["falsos_positivos"], label="Falsos positivos", color="#4C78A8")
ax.bar(por_anio["grupo"], por_anio["falsos_negativos"], bottom=por_anio["falsos_positivos"],
       label="Falsos negativos", color="#E45756")
ax.set(title="Errores del HistGradientBoosting por año (umbral fijo 0.50)",
       xlabel="Año del préstamo", ylabel="Cantidad")
ax.legend()
fig.tight_layout()
fig_dir = RAIZ / "reports/figures/analisis_errores"
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "01_errores_por_anio.png", dpi=160)
plt.show()

In [ ]:
por_plazo = hgb[hgb["segmento"] == "plazo"].sort_values("grupo")
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(por_plazo["grupo"], por_plazo["recall_umbral_0_5"], color="#59A14F")
ax.set(title="Detección de incumplimientos por plazo", xlabel="Plazo (meses)",
       ylabel="Recall con umbral fijo 0.50", ylim=(0, 1))
fig.tight_layout()
fig.savefig(fig_dir / "02_recall_por_plazo.png", dpi=160)
plt.show()

## 3. Lectura e interpretación disponible

La interpretación principal en este corte es descriptiva: comparar errores, recall y precisión entre años, plazos, rangos de ingreso y tipos de solicitud. El repositorio no guarda el objeto entrenado del modelo final ni una importancia por permutación calculada con el mismo pipeline final. Por eso, los coeficientes logísticos y la importancia del árbol disponibles en `outputs/` se consideran evidencia histórica de modelos anteriores, no una explicación del HistGradientBoosting final.

Para afirmar diferencias consistentes entre grupos haría falta revisar el tamaño de cada grupo, su intervalo de incertidumbre y posibles variables omitidas. Tampoco se dispone de atributos sensibles suficientes para una auditoría de equidad completa.

## 4. Limitaciones

- La evaluación incluye únicamente préstamos con estado final observado. En 2017–2018 muchos préstamos del archivo seguían activos; los resultados no representan necesariamente todas las solicitudes de esos años.
- El archivo no contiene la fecha exacta en que se conoció cada desenlace. La maduración temporal de los préstamos no se puede reconstruir por completo.
- El umbral 0.50 se presenta solo para describir decisiones; no representa una regla aprobada para otorgar o rechazar crédito.
- Los resultados por segmento muestran asociaciones y diferencias de desempeño, no causalidad. La cobertura de `Fully Paid` y `Charged Off` puede variar entre segmentos.